# Using Gemini for Video Analysis

Analyse a video with Google's native Python SDK, ask questions with timestamped evidence, and extract validated structured outputs.

This is the practical companion to **Using Gemini for Video Analysis**. Follow the deck's sequence: **upload once, wait for ACTIVE, then ask**. We use `google-genai` and the Interactions API throughout.

By the end, you can reuse one uploaded video, continue a conversation, turn observations into a Pydantic model, and choose a sampling strategy for your task.

## 1. Why Gemini is competitive for video

Gemini accepts a video file and processes its visual and audio content together. That makes questions such as "What happens while the presenter explains this step?" straightforward to implement.

| Provider and API | Route to video analysis | Work your application handles |
| --- | --- | --- |
| Gemini Interactions | Send the video directly | Upload and reference the file, choose processing options |
| OpenAI Responses with GPT-6.1 Sol | Send images of selected frames and any transcript | Extract frames, preserve timestamps, obtain and align a transcript |
| Claude Messages vision | Send images of selected frames and any transcript | Extract frames, preserve timestamps, obtain and align a transcript |

GPT-6.1 Sol's model page lists image input, with audio and video unsupported. Claude's vision guide describes image inputs. Both can reason about supplied frames, but their APIs in this comparison need preprocessing. This comparison concerns these developer APIs; consumer apps and agent tools can provide different workflows.

Sources: [Gemini video](https://ai.google.dev/gemini-api/docs/video-understanding), [GPT-6.1 Sol modalities](https://developers.openai.com/api/docs/models/gpt-6.1-sol), [OpenAI vision](https://developers.openai.com/api/docs/guides/images-vision), [Claude vision](https://platform.claude.com/docs/en/build-with-claude/vision).

### Price is one part of the decision

Standard paid API rates in USD per million tokens, checked **8 October 2026**:

| Model | Uncached input | Output |
| --- | ---: | ---: |
| Gemini 3.8 Flash | $0.75 | $3.75 |
| GPT-6.1 Sol | $2.00 | $10.00 |
| Claude Sonnet 5.5 | $2.00 | $10.00 |
| Claude Haiku 5.5, prompts up to 100,000 tokens | $0.10 | $0.50 |

Gemini's listed promotional rates end on 31 December 2026, then rise to USD 1.50 input and USD 7.50 output. Haiku's rates rise above the stated prompt threshold. These prices are **not an equal-quality benchmark or a cost-per-video comparison**: providers tokenise media differently, reasoning consumes tokens, and caching changes cost. OpenAI also changes rates for long prompts.

Our practical conclusion: Gemini is a strong starting point when native video input saves preprocessing and its answers pass your checks. For a handful of still images or an existing transcript, another model may cost less. Measure correctness, latency and the cost of the complete workflow on your own clips.

Sources: [Google pricing](https://ai.google.dev/gemini-api/docs/pricing), [OpenAI model pricing](https://developers.openai.com/api/docs/models/gpt-6.1-sol), [Claude pricing](https://platform.claude.com/docs/en/about-claude/pricing).

## 2. Why Google's native SDK

The [Files API](https://ai.google.dev/gemini-api/docs/files) stores uploads for **48 hours**, with a **2 GB per-file** limit. Reuse the URI across questions instead of sending the file bytes each time. Google recommends the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions-overview) for new projects, and it supports conversations through `previous_interaction_id`.

There are three separate kinds of reuse:

| Mechanism | What it saves or preserves |
| --- | --- |
| Uploaded file URI | Repeated transfer of the video bytes |
| `previous_interaction_id` | Conversation history on the server |
| Prompt caching | Inference work and token charges when a cache actually applies |

**One upload does not mean one input-token charge.** Each analysis can process video or conversation context again. Look at the returned usage.

A gateway can also support video. For example, [OpenRouter documents video URLs and base64 input](https://openrouter.ai/docs/guides/overview/multimodal/videos). Use a gateway for a shared provider interface when it supports your needs. Here the native SDK exposes file management and Gemini's conversation and processing controls directly.

## 3. Setup

Run in **Jupyter or Google Colab with Python 3.10+**. Create a [Gemini API key in Google AI Studio](https://aistudio.google.com/apikey). We use a hidden `getpass` prompt if `GEMINI_API_KEY` is missing.

The main lesson makes **four generation requests**: a summary, a follow-up, a structured timeline, and structured Q&A. Paid accounts incur charges. One optional dense-sampling request starts disabled. No OpenAI or Anthropic key is needed.

The SDK version below keeps the demonstrated API reproducible. If you change it, check the linked documentation and restart your kernel if packages were already imported.

In [ ]:
"""Analyse video with Gemini and validate timestamped, structured answers."""

# Pylint treats notebook cells as one script and replaces pip magics with pass.
# pylint: disable=wrong-import-position,unnecessary-pass
%pip install "google-genai==2.29.0" "pydantic>=2.9,<3" pandas --upgrade --quiet

In [ ]:
MODEL = "gemini-3.8-flash"

# Leave empty to download Google's public pottery sample, about 12 seconds / 70 MB.
# Or set a path to an MP4 in your Jupyter or Colab runtime.
VIDEO_PATH = ""
SAMPLE_URL = "https://storage.googleapis.com/generativeai-downloads/videos/Pottery.mp4"

RUN_DENSE_SAMPLING = False

In [ ]:
import json
import os
import shutil
import tempfile
import time
from getpass import getpass
from pathlib import Path
from typing import Literal
from urllib.request import urlopen

import pandas as pd
from google import genai
from google.genai.interactions import Interaction
from IPython.display import Markdown, Video, display
from pydantic import BaseModel, Field, model_validator

if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

INSTRUCTIONS = (
    "Analyse only the supplied video. Separate visible evidence from audible evidence. "
    "Use MM:SS timestamps for prose answers. State when something cannot be observed. "
    "Avoid guessing identities, intentions or events outside the video."
)
GENERATION_CONFIG = {"max_output_tokens": 4096, "thinking_level": "low"}
requests_made: list[tuple[str, Interaction]] = []

## 4. Choose and watch the video

The default downloads a short public pottery clip hosted by Google. Watch it before asking the model questions so you can check its answers. Replace `VIDEO_PATH` in the configuration cell to analyse your own recording. In Colab, use the Files sidebar to upload an MP4, then copy its runtime path into that cell.

The sample preview streams the public URL. For your own file the preview embeds its bytes in the cell output, so use a short clip and clear that output before sharing the notebook.

In [ ]:
if VIDEO_PATH:
    video_path = Path(VIDEO_PATH).expanduser()
else:
    # Store the sample in the runtime's temporary directory, outside the repository.
    video_path = Path(tempfile.gettempdir()) / "gemini_video_analysis_pottery.mp4"
    if not video_path.exists():
        print("Downloading the public sample...")
        temporary_path = video_path.with_suffix(".download")
        try:
            with urlopen(SAMPLE_URL, timeout=60) as source, temporary_path.open("wb") as target:
                shutil.copyfileobj(source, target)
            temporary_path.replace(video_path)
        finally:
            temporary_path.unlink(missing_ok=True)

if not video_path.is_file() or video_path.stat().st_size == 0:
    raise FileNotFoundError(f"Choose an existing, non-empty video: {video_path}")
if video_path.suffix.lower() != ".mp4":
    raise ValueError("This introductory example expects an MP4 file.")
if video_path.stat().st_size > 2 * 1024**3:
    raise ValueError("The Files API has a 2 GB per-file limit. Use a shorter MP4.")

print(f"{video_path.name}: {video_path.stat().st_size / 1_000_000:.1f} MB")
if VIDEO_PATH:
    display(Video(filename=str(video_path), embed=True, width=640))
else:
    display(Video(url=SAMPLE_URL, width=640))

## 5. Upload once, then wait for ACTIVE

An upload can return while Google's video processing is still running. The loop below waits for `ACTIVE`, reports failed processing, and has a five-minute deadline. After a timeout you can retrieve the existing file by its name and check its state again.

In [ ]:
video_file = client.files.upload(
    file=str(video_path),
    config={"mime_type": "video/mp4", "display_name": video_path.name},
)
video_file_name = video_file.name
if not video_file_name:
    raise RuntimeError("The upload did not return a file name.")
print("Uploaded:", video_file_name)

deadline = time.monotonic() + 300
while video_file.state and video_file.state.name == "PROCESSING":
    if time.monotonic() >= deadline:
        raise TimeoutError(
            f"Processing still running. Check client.files.get(name={video_file_name!r})."
        )
    time.sleep(2)
    video_file = client.files.get(name=video_file_name)

if not video_file.state or video_file.state.name != "ACTIVE":
    raise RuntimeError(f"Video is not ready: {video_file.state}. Details: {video_file.error}")

print("Ready:", video_file.state.name)
video_input = {
    "type": "video",
    "uri": video_file.uri,
    "mime_type": video_file.mime_type,
    "processing": "static",
    "resolution": "low",
}

## 6. Ask a question with timestamps

Static processing samples **one frame per second by default**, alongside audio. At low resolution Google estimates roughly **100 tokens per second**, so a ten-minute video can use around 60,000 input tokens before the prompt and answer. Fast actions can fall between sampled frames.

Place the video before the question. The response's `output_text` contains the answer. We explicitly keep this interaction so the next cell can continue the conversation.

Source: [Video processing and tokens](https://ai.google.dev/gemini-api/docs/video-understanding).

In [ ]:
summary = client.interactions.create(
    model=MODEL,
    system_instruction=INSTRUCTIONS,
    generation_config=GENERATION_CONFIG,
    input=[
        video_input,
        {"type": "text", "text": (
            "Summarise the clip in three short bullets. Cite a MM:SS timestamp for each. "
            "Describe the visible actions and mention any audio you can actually hear."
        )},
    ],
    store=True,
    stream=False,
)
if not isinstance(summary, Interaction):
    raise TypeError("Expected a non-streaming Interaction response.")
requests_made.append(("Summary", summary))
summary_id = summary.id
if not summary_id:
    raise RuntimeError("The stored summary did not return an interaction ID.")
print("Stored interaction:", summary_id)

if summary.status != "completed" or not summary.output_text:
    raise RuntimeError(
        f"No completed answer: status={summary.status}, errors={summary.errors}"
    )
display(Markdown(summary.output_text))

### Follow-up Q&A with conversation history

`previous_interaction_id` points to the previous turn. You can ask a follow-up without supplying the video again. Repeat your instructions and generation settings: those apply to each interaction individually.

The history can still contribute to token usage. For an independent question, send `[video_input, question]` again using the same uploaded URI.

Source: [Interactions conversation state](https://ai.google.dev/gemini-api/docs/interactions-overview).

In [ ]:
follow_up = client.interactions.create(
    model=MODEL,
    system_instruction=INSTRUCTIONS,
    generation_config=GENERATION_CONFIG,
    previous_interaction_id=summary_id,
    input="What changes between the beginning and the end? Cite moments that support your answer.",
    store=False,
    stream=False,
)
if not isinstance(follow_up, Interaction):
    raise TypeError("Expected a non-streaming Interaction response.")
requests_made.append(("Follow-up Q&A", follow_up))

if follow_up.status != "completed" or not follow_up.output_text:
    raise RuntimeError(
        f"No completed answer: status={follow_up.status}, errors={follow_up.errors}"
    )
display(Markdown(follow_up.output_text))

## 7. Structured outputs: video into Python data

A prose summary helps a reader. A typed timeline lets your application build chapters, search observations or prepare a review table.

Define the expected shape with Pydantic, pass `model_json_schema()` as the API's `response_format`, then validate the returned JSON with `model_validate_json()`. The `model_validator` checks that an event ends after it starts. The API schema alone cannot enforce that relationship.

We use numeric seconds for data and convert them to MM:SS for display. `audio_evidence` and `uncertainty` accept `None`. A schema can enforce shape and types; **it cannot establish that an observation is true**. Watch the cited moments to verify them.

Source: [Gemini structured outputs](https://ai.google.dev/gemini-api/docs/structured-output). Gemini supports a subset of JSON Schema, so keep schemas small.

In [ ]:
class VideoEvent(BaseModel):
    """A timestamped interval with visible and audible evidence."""

    start_seconds: float = Field(
        ge=0, description="Event start, in seconds from the original video's start."
    )
    end_seconds: float = Field(
        ge=0, description="Event end, in seconds from the original video's start."
    )
    visual_evidence: str = Field(description="What is directly visible during this interval.")
    audio_evidence: str | None = Field(
        description="What is directly audible, or null if there is no relevant audio."
    )
    uncertainty: str | None = Field(description="Any uncertainty in the observation, or null.")

    @model_validator(mode="after")
    def check_interval(self) -> "VideoEvent":
        """Reject intervals that do not end after they start."""
        if self.end_seconds <= self.start_seconds:
            raise ValueError("Each event must end after it starts.")
        return self


class VideoAnalysis(BaseModel):
    """A short summary and a bounded timeline of video events."""

    title: str
    summary: str
    events: list[VideoEvent] = Field(min_length=1, max_length=5)


structured = client.interactions.create(
    model=MODEL,
    system_instruction=INSTRUCTIONS,
    generation_config=GENERATION_CONFIG,
    input=[video_input, {"type": "text", "text": (
        "Extract a short title, a one-sentence summary and 1 to 5 key events. "
        "Use seconds relative to the original video. Every interval must fit within the video. "
        "Separate visual from audio evidence, and record uncertainty."
    )}],
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": VideoAnalysis.model_json_schema(),
    },
    store=False,
    stream=False,
)
if not isinstance(structured, Interaction):
    raise TypeError("Expected a non-streaming Interaction response.")
requests_made.append(("Structured timeline", structured))

if structured.status != "completed" or not structured.output_text:
    raise RuntimeError(
        f"No completed JSON: status={structured.status}, errors={structured.errors}"
    )
analysis = VideoAnalysis.model_validate_json(structured.output_text)
print(analysis.model_dump_json(indent=2))

In [ ]:
def timestamp(seconds: float) -> str:
    """Convert seconds to an MM:SS timestamp for display."""
    minutes, remaining = divmod(int(seconds), 60)
    return f"{minutes:02d}:{remaining:02d}"


timeline = pd.DataFrame([
    {
        "Start": timestamp(event.start_seconds),
        "End": timestamp(event.end_seconds),
        "Visible evidence": event.visual_evidence,
        "Audio evidence": event.audio_evidence,
        "Uncertainty": event.uncertainty,
    }
    for event in analysis.events
])
display(timeline)

## 8. Structured Q&A with an explicit unknown

Now ask one question that the clip should answer and one about an event outside the recording. This tests whether the model can acknowledge missing evidence.

Use a small contract: the answer, a status, and timestamped evidence. A supported answer needs at least one evidence item. An unobservable answer should explain what the video cannot establish. We also check that the model answered the requested questions.

In [ ]:
class Evidence(BaseModel):
    """A timestamped observation supporting an answer."""

    timestamp_seconds: float = Field(ge=0)
    modality: Literal["visual", "audio", "both"]
    observation: str


class VideoAnswer(BaseModel):
    """An answer that records whether the video supports it."""

    question: str
    answer: str
    status: Literal["supported", "not_observable"]
    evidence: list[Evidence]

    @model_validator(mode="after")
    def check_support(self) -> "VideoAnswer":
        """Require timestamped evidence for supported answers."""
        if self.status == "supported" and not self.evidence:
            raise ValueError("A supported answer needs timestamped evidence.")
        return self


class VideoQA(BaseModel):
    """Structured answers to the requested video questions."""

    answers: list[VideoAnswer]


QUESTIONS = [
    "What does the main person or object do in this clip?",
    "What exact price did the item sell for after the recording ended?",
]

qa_response = client.interactions.create(
    model=MODEL,
    system_instruction=INSTRUCTIONS,
    generation_config=GENERATION_CONFIG,
    input=[video_input, {"type": "text", "text": (
        "Answer each question below in order, copying each question exactly. "
        "For supported answers, give timestamped evidence. For anything the video "
        "cannot establish, use not_observable and explain the missing evidence. "
        "Do not use outside knowledge to fill gaps.\n" + json.dumps(QUESTIONS)
    )}],
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": VideoQA.model_json_schema(),
    },
    store=False,
    stream=False,
)
if not isinstance(qa_response, Interaction):
    raise TypeError("Expected a non-streaming Interaction response.")
requests_made.append(("Structured Q&A", qa_response))

if qa_response.status != "completed" or not qa_response.output_text:
    raise RuntimeError(
        f"No completed JSON: status={qa_response.status}, errors={qa_response.errors}"
    )
qa = VideoQA.model_validate_json(qa_response.output_text)
if [answer.question for answer in qa.answers] != QUESTIONS:
    raise ValueError("The output does not match the requested questions and order.")

display(pd.DataFrame([
    {
        "Question": answer.question,
        "Answer": answer.answer,
        "Status": answer.status,
        "Evidence": "; ".join(
            f"{timestamp(item.timestamp_seconds)} ({item.modality}): {item.observation}"
            for item in answer.evidence
        ),
    }
    for answer in qa.answers
]))
print("Outside-video question marked unknown:", qa.answers[1].status == "not_observable")

## 9. Optional: inspect a short interval more closely

Enable `RUN_DENSE_SAMPLING` and rerun this cell to inspect the first five seconds at three frames per second. This adds one request. More samples may capture a brief movement, and also consume more tokens. Use static processing for explicit clipping and FPS controls. In this SDK, offsets use duration strings such as `"5s"`.

For another clip, choose a relevant interval inside its duration. Compare against section 6 and replay the interval yourself. Higher FPS does not guarantee accurate timing.

In [ ]:
if RUN_DENSE_SAMPLING:
    detailed_input = {
        **video_input,
        "processing": {"type": "static", "start_offset": "0s", "end_offset": "5s", "fps": 3},
    }
    detailed = client.interactions.create(
        model=MODEL,
        system_instruction=INSTRUCTIONS,
        generation_config=GENERATION_CONFIG,
        input=[detailed_input, {"type": "text", "text": (
            "Describe the visible actions in this interval in their order. "
            "Use timestamps relative to the original video and state any uncertainty."
        )}],
        store=False,
        stream=False,
    )
    if not isinstance(detailed, Interaction):
        raise TypeError("Expected a non-streaming Interaction response.")
    requests_made.append(("Dense five-second interval", detailed))
    if detailed.status != "completed" or not detailed.output_text:
        raise RuntimeError(
            f"No completed answer: status={detailed.status}, errors={detailed.errors}"
        )
    display(Markdown(detailed.output_text))
else:
    print("Dense sampling is disabled. Set RUN_DENSE_SAMPLING = True to try it.")

## 10. Inspect the actual token usage

The Interactions API returns `usage`. Compare the summary and follow-up to see that a shorter request payload does not mean zero input tokens. Cached tokens are a subset of input tokens, so avoid adding them to the input count again. Thought and tool-use fields help explain the work behind an answer.

This table shows each response's reported counts. Use your provider's billing view for the final charge. For model selection, record full-workflow cost, latency, correctly answered questions, false claims and timestamp accuracy on the same test clips.

In [ ]:
usage_rows = []
for label, interaction in requests_made:
    usage = interaction.usage
    usage_rows.append({
        "Request": label,
        "Input tokens": usage.total_input_tokens if usage else None,
        "Cached tokens": usage.total_cached_tokens if usage else None,
        "Output tokens": usage.total_output_tokens if usage else None,
        "Thought tokens": usage.total_thought_tokens if usage else None,
        "Tool-use tokens": usage.total_tool_use_tokens if usage else None,
        "Total tokens": usage.total_tokens if usage else None,
    })
display(pd.DataFrame(usage_rows))

## 11. Clean up your uploaded video

Run this after all questions. It deletes only this lesson's uploaded file. To analyse it again after closing the client, rerun the setup and upload cells. The Files API otherwise expires it automatically after 48 hours.

File storage and interaction history are separate. The first summary uses a stored interaction. We delete it too after finishing with the conversation.

In [ ]:
client.files.delete(name=video_file_name)
print("Deleted the lesson upload:", video_file_name)

# Delete the interaction we explicitly stored for conversation support.
client.interactions.delete(summary_id)
print("Deleted the stored lesson interaction.")
client.close()

## Practice

1. Replace the sample with a short screen recording or lesson clip. Ask a question requiring both spoken and visual evidence.
2. Add a `chapter_title` field to `VideoEvent`, request it in the prompt and rerun the structured-output cells.
3. Write five answerable questions and two questions the recording cannot establish. Watch the clip, write reference answers, then check Gemini's evidence and unknown handling.
4. Compare one frame per second with three frames per second on the same interval. Record correctness, latency and usage before choosing a sampling rate.

### Troubleshooting

| Symptom | What to check |
| --- | --- |
| Invalid API key | Use a current Gemini key from AI Studio |
| Model unavailable | Check your account's access and the current model documentation, then update `MODEL` |
| Quota or rate-limit error | Check quota and billing in AI Studio, then wait before retrying |
| File expired or missing | Rerun upload and use its new URI |
| Empty text or incomplete response | Inspect `status` and `errors`; check output limits and the supplied media |
| Pydantic validation error | Inspect the JSON, clarify the prompt and keep the schema small |
| Important action missing | Inspect a shorter interval with denser sampling, then compare to the actual video |

## Summary

- Gemini's direct video input reduces the frame extraction and audio alignment your application needs to perform.
- The native SDK provides Files API uploads, conversation state and video processing controls.
- Upload once, wait for `ACTIVE`, then reference the file in your questions. Transfer reuse, conversation history and caching are distinct.
- Timestamped Q&A and typed outputs make answers easier to inspect and use. Schema validation still needs evidence checks.
- Choose models using the complete task's accuracy, latency and cost. Sampling and missing evidence affect what any answer can establish.